In [30]:
import pandas as pd

# Load saved training and test CSV files from week 3
train = pd.read_csv("data/cleaned_train.csv")
test = pd.read_csv("data/cleaned_test.csv")

print("Training shape: ", train.shape)
print("Test shape: ", test.shape)

Training shape:  (117914, 828)
Test shape:  (12789, 828)


In [ ]:
target = "ClosePrice"

city_cols = [
    col for col in train.columns
    if col.startswith("City_grouped_")
]

postal_cols = [
    col for col in train.columns
    if col.startswith("PostalCode_grouped_")
]

county_cols = [
    col for col in train.columns 
    if col.startswith("CountyOrParish_")
]

school_district_cols = [
    col for col in train.columns 
    if col.startswith("SchoolDistrict_")
]

features_sets = {
    "Basic": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet"
        ],

    "With Property Features": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ],

    "With Missing LotSize Flagged": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ],

    "With Location Features (City Only)": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
    ] + city_cols,

    "With Location Features (PostalCode Only)": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ] + postal_cols,

    "With Location Features (County Only)": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ] + county_cols,

    "With All Location Features": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN"
    ] + city_cols + postal_cols + county_cols,

    "With Engineered Features": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
        "PropertyAge",
        "BedBathRatio"
    ],

    "With Engineered Features + All Location + SchoolDistrict": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
        "PropertyAge",
        "BedBathRatio"
    ]    + city_cols + postal_cols + county_cols + school_district_cols,

    "With Everything + Missing_YearBuilt": [
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "Missing_LotSizeSquareFeet",
        "ViewYN",
        "WaterfrontYN",
        "BasementYN",
        "PoolPrivateYN",
        "PropertyAge",
        "Missing_YearBuilt",
        "BedBathRatio"
    ]    + city_cols + postal_cols + county_cols + school_district_cols

}


In [32]:
# Ensure no missing values
'''for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Ensure no missing values
    print(f"\n{name}")
    print("----------------------")
    missing = X_train.columns[X_train.isna().any()]     # Training Set
    print(missing)
    for col in missing:
        print(col, X_train[col].isna().sum())

    missing = X_test.columns[X_test.isna().any()]       # Test Set
    print(missing)
    for col in missing:
        print(col, X_test[col].isna().sum())

    print("\nTarget Missing")
    print("----------------------")
    print("Train:", y_train.isnull().sum())
    print("Test:", y_test.isnull().sum())'''

'for name, features, in features_sets.items():\n\n    # Define features and target\n    X_train = train[features]   # Training Set\n    y_train = train[target]\n\n    X_test = test[features]     # Test Set\n    y_test = test[target]\n\n    # Ensure no missing values\n    print(f"\n{name}")\n    print("----------------------")\n    missing = X_train.columns[X_train.isna().any()]     # Training Set\n    print(missing)\n    for col in missing:\n        print(col, X_train[col].isna().sum())\n\n    missing = X_test.columns[X_test.isna().any()]       # Test Set\n    print(missing)\n    for col in missing:\n        print(col, X_test[col].isna().sum())\n\n    print("\nTarget Missing")\n    print("----------------------")\n    print("Train:", y_train.isnull().sum())\n    print("Test:", y_test.isnull().sum())'

In [33]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# Linear Regression Results from Week 4
linear_results = []

for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = LinearRegression()      # Initialize linear regression model as baseline
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Save results
    linear_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 6),
        "Test R^2": round(r2_test, 6)
    })

    '''# Print results
    print(f"{name}")
    print("-----------------------------------")
    print(f"Training R^2: {r2_train:.4f}")
    print(f"Test R^2: {r2_test:.4f}")
    print()'''

In [34]:
linear_results_df = pd.DataFrame(linear_results)
linear_results_df

,Feature Set,Number of Features,Training R^2,Test R^2
0,Basic,4,0.363923,0.371086
1,With Property Features,8,0.363923,0.371086
2,With Missing LotSize Flagged,9,0.363923,0.371086
3,With Location Features (City Only),209,0.363923,0.371086
4,With Location Features (PostalCode Only),209,0.363923,0.371086
5,With Location Features (County Only),68,0.363923,0.371086
6,With All Location Features,470,0.363923,0.371086
7,With Engineered Features,11,0.427356,0.440744
8,With Engineered Features + All Location,795,0.427374,0.440762
9,With Engineered Features + Unknown YearBuilt +...,796,0.427374,0.440762


Baseline Linear Regression Model Results

From Week 5:
- The model generalizes well for all feature sets version.
- But the R^2 score is relatively low, suggesting possible underfitting.

From Week 6:
- R^2 scores higher with the sample features included (0.364->0.427).
- Test results slightly higher than that of training (~0.1-0.2 difference).

In [35]:
# Decision Tree Regressor
decision_tree_results = []

for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = DecisionTreeRegressor(random_state=24)      # Initialize linear regression model as baseline
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Save results
    decision_tree_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 6),
        "Test R^2": round(r2_test, 6)
    })


In [36]:
decision_tree_results = pd.DataFrame(decision_tree_results)
decision_tree_results

,Feature Set,Number of Features,Training R^2,Test R^2
0,Basic,4,0.995643,-0.016423
1,With Property Features,8,0.997932,-0.023133
2,With Missing LotSize Flagged,9,0.997934,-0.028392
3,With Location Features (City Only),209,0.999359,0.436744
4,With Location Features (PostalCode Only),209,0.998803,0.296635
5,With Location Features (County Only),68,0.999097,0.335991
6,With All Location Features,470,0.999426,0.615222
7,With Engineered Features,11,0.999707,0.104717
8,With Engineered Features + All Location,795,0.999747,0.650386
9,With Engineered Features + Unknown YearBuilt +...,796,0.999747,0.652362


Decision Tree Regressor Results

From Week 5:
- Explained most variablity when included all three location features in addition to the five key variables and flagged missing lot size.
- Does not generalize well for any of the feature sets used; the most comprehensive feature set has the least difference in R^2 score between test and training set, but >0.3.
- Very high R^2 from the training data, suggesting overfitting.

From Week 6:
- Least gap between training and test results with new sample features engineered. 
- Still suggests overfitting as observed in the really high R^2 from training set.

In [37]:
# Random Forest Regressor
random_forest_results = []

for name, features, in features_sets.items():

    # Define features and target
    X_train = train[features]   # Training Set
    # print(name, X_train.shape)
    y_train = train[target]

    X_test = test[features]     # Test Set
    y_test = test[target]

    # Model
    model = RandomForestRegressor(random_state=24, n_estimators=100, n_jobs=1)      # Initialize linear regression model as baseline
    model.fit(X_train, y_train)     # Train model

    # Predictions on target variable
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Compute R^2 scores
    r2_train = r2_score(y_train, y_pred_train)
    r2_test = r2_score(y_test, y_pred_test)

    # Save results
    random_forest_results.append({
        "Feature Set": name,
        "Number of Features": len(features),
        "Training R^2": round(r2_train, 6),
        "Test R^2": round(r2_test, 6)
    })


In [38]:
random_forest_results = pd.DataFrame(random_forest_results)
random_forest_results

,Feature Set,Number of Features,Training R^2,Test R^2
0,Basic,4,0.914534,0.414912
1,With Property Features,8,0.918425,0.428453
2,With Missing LotSize Flagged,9,0.918481,0.428769
3,With Location Features (City Only),209,0.954289,0.677608
4,With Location Features (PostalCode Only),209,0.942693,0.601145
5,With Location Features (County Only),68,0.947831,0.630628
6,With All Location Features,470,0.969240,0.776119
7,With Engineered Features,11,0.932925,0.531352
8,With Engineered Features + All Location,795,0.974161,0.810480
9,With Engineered Features + Unknown YearBuilt +...,796,0.974167,0.810361


Random Forest Regressor Results

From Week 5:
- Explained the most variability and least difference in R^2 score between test and training set on the most comprehensive feature set.
- Clear improvement in model performace after adding location features.
- Using only the city information explianed slightly more variablity than using either the postal code or county alone.

From Week 6:
- Shifting the dataset used forward by one month gave similar results, slightly narrowed the gap difference between training and test.
- Adding sample features (PropertyAge, BedBathRatio, SchoolDistrict) further improved the model.
- Explained slightly more variability
    - Training: 0.970->0.974
    - Test: 0.776->0.810

Comparison Summary
- compare test R^2 from the three models.
- applied on all of the feature sets the same way as with the baseline model for comparison on model performances.

In [39]:
test_compare_df = (
    linear_results_df[["Feature Set", "Test R^2"]]
    .rename(columns={"Test R^2": "Baseline Linear Regression"})
    .merge(
        decision_tree_results[["Feature Set", "Test R^2"]]
        .rename(columns={"Test R^2": "Decision Tree Regressor"}),
        on="Feature Set"
    )
    .merge(
        random_forest_results[["Feature Set", "Test R^2"]]
        .rename(columns={"Test R^2": "Random Forest Regressor"}),
        on="Feature Set"
    )
)
test_compare_df

,Feature Set,Baseline Linear Regression,Decision Tree Regressor,Random Forest Regressor
0,Basic,0.371086,-0.016423,0.414912
1,With Property Features,0.371086,-0.023133,0.428453
2,With Missing LotSize Flagged,0.371086,-0.028392,0.428769
3,With Location Features (City Only),0.371086,0.436744,0.677608
4,With Location Features (PostalCode Only),0.371086,0.296635,0.601145
5,With Location Features (County Only),0.371086,0.335991,0.630628
6,With All Location Features,0.371086,0.615222,0.776119
7,With Engineered Features,0.440744,0.104717,0.531352
8,With Engineered Features + All Location,0.440762,0.650386,0.810480
9,With Engineered Features + Unknown YearBuilt +...,0.440762,0.652362,0.810361


From Week 5:
- The Decision Tree Regressor Model has unstable performance. It only outperformed the Linear Regression Model when the city location feature was included (R^2 = 0.441) or when all three location features were included (R^2 = 0.607) in addition the the 5 key features.

- The Random Forest Regressor Model consistently outperform the Linear Regression Model, with a highest R^2 score of 0.770 with the test set that includes all location features.

From Week 6 (with train/test data months update):
- Have improvement overall based on R^2 scores.

- The Decision Tree Regressor Model is still unstable. Only outperformed the Linear Regression Model when either the city feature or all location features were included, and when the engineered and all locations features were included.

- The Random Forest Regressor Model consistently outperformed both the other two models under all feature sets tested.

Model Strengths and Weaknesses

| Model | Strengths | Weaknesses |
| ----- | --------- | ---------- |
| Linear Regression (Baseline) | Stable performance, fast, generalize well | Cannot capture nonlinear or complex patterns between property features |
| Decision Tree Regressor | Can capture more complex relationship between features (eg location features interaction with the key varibales) | Unstable performance, highly prone to overfitting |
| Random Forest Regressor | Can capture more complex relationship between features, most consistent outperformance than both the other models, less prone to overfitting than Decision Tree Regressor | Slower to compute |